# Embedding des données

In [1]:
import pandas as pd

from src.backend.vector_store import explode_chunks, build_vector_store
from src.backend.pgvector_store import upsert_chunks

/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1) Charger le CSV enrichi avec rag_document
csv_path = "../data/processed/parcoursup_2026_enriched_chunks.csv"  # adapte au bon chemin

df = pd.read_csv(csv_path)
df.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,...,formation_ouverte_boursiers,diplome_controle_par_etat,formation_selective,epreuves_selection,frais_candidature,frais_candidature_boursiers,rag_document,chunk_index,chunk_text,chunk_id
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €",Nom de la formation : Formation d'ingénieur Ba...,0,Nom de la formation : Formation d'ingénieur Ba...,0_0
1,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €",Nom de la formation : Formation d'ingénieur Ba...,1,présenter un projet.\nDisposer de compétences...,0_1
2,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €",Nom de la formation : Formation d'ingénieur Ba...,2,.\nLes diplômés Polytech exercent dans des sec...,0_2
3,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €",Nom de la formation : Formation d'ingénieur Ba...,0,Nom de la formation : Formation d'ingénieur Ba...,1_0
4,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €",Nom de la formation : Formation d'ingénieur Ba...,1,t présenter un projet.\nDisposer de compétence...,1_1


In [3]:
# 2) Créer la base vectorielle (embeddings e5) pour tous les chunks
df_vs = build_vector_store(df)

df_vs.head()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9737.24it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,chunk_id,chunk_index,chunk_text,type_formation,type_etablissement,commune,is_apprentissage,frais_scolarite,formation_selective,formation_ouverte_boursiers,embedding
0,0_0,0,Nom de la formation : Formation d'ingénieur Ba...,Formations des écoles d'ingénieurs,Publics,Dijon,0,628 €,1.0,1.0,"[0.009988975, 0.04926425, -0.010969478, 0.0157..."
1,0_1,1,présenter un projet.\nDisposer de compétences...,Formations des écoles d'ingénieurs,Publics,Dijon,0,628 €,1.0,1.0,"[0.0021161127, 0.02797542, -0.010582869, 0.008..."
2,0_2,2,.\nLes diplômés Polytech exercent dans des sec...,Formations des écoles d'ingénieurs,Publics,Dijon,0,628 €,1.0,1.0,"[-0.0042433487, 0.050291143, -0.018544631, 0.0..."
3,1_0,0,Nom de la formation : Formation d'ingénieur Ba...,Formations des écoles d'ingénieurs,Publics,Angers,0,628 €,1.0,1.0,"[0.007669793, 0.048459146, -0.011819439, 0.023..."
4,1_1,1,t présenter un projet.\nDisposer de compétence...,Formations des écoles d'ingénieurs,Publics,Angers,0,628 €,1.0,1.0,"[0.0011799281, 0.022003515, -0.011385879, 0.00..."


# Insertion dans la base vectorielle

In [4]:
# 3) Insérer tous les chunks et embeddings dans Postgres/pgvector
upsert_chunks(df_vs)

In [5]:
# 4) (Optionnel) Afficher quelques lignes depuis Postgres pour vérifier
from src.backend.pgvector_store import get_pg_connection

conn = get_pg_connection()
cur = conn.cursor()
cur.execute("SELECT chunk_id, chunk_index, LEFT(chunk_text, 80) FROM formations_chunks_vectors LIMIT 5;")
print(cur.fetchall())
cur.close()
conn.close()

[('47_0', 0, 'Nom de la formation : BTS - Production - Conception et réalisation en chaudronne'), ('48_0', 0, 'Nom de la formation : BUT - Techniques de commercialisation\n\nType de formation :'), ('14076_1', 1, ' en apprentissage.\nChangeons la donne pour un numérique en mode Humanware. Faiso'), ('0_0', 0, "Nom de la formation : Formation d'ingénieur Bac + 5 - Bac général\n\nType de forma"), ('0_1', 1, ' présenter un projet.\nDisposer de compétences écrites et orales en langues étran')]
